# SRT & HRT: how does solids retention time affect effluent quality?

This notebook lets you interactively explore how **solids retention time (SRT)** affects effluent quality in an activated sludge process, using [QSDsan](https://qsdsan.com)'s steady-state `ActivatedSludgeProcess` model (Rittmann & McCarty kinetics).

**Concept:** SRT is the average time the biomass (the microorganisms that degrade the waste) spends in the system. If SRT is too short, biomass is washed out of the system faster than it can grow, and treatment collapses — there is a theoretical minimum SRT (`SRT_min`) below which this always happens, no matter how large the reactor. Above `SRT_min`, effluent quality improves as SRT increases, with diminishing returns.

**Try it:** run all cells, then drag the **Target SRT** slider down until the system washes out, then back up and watch effluent COD fall and then flatten out.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import ipywidgets as widgets
import matplotlib.pyplot as plt

from qsdsan_teaching.systems import simulate_asp, effluent_cod_curve
from qsdsan_teaching.widgets import labeled_slider, format_readout, format_impact_comparisons
from qsdsan_teaching.plotting import plot_srt_curve

%matplotlib inline

In [ ]:
srt_slider = labeled_slider('Target SRT (d)', value=5, min=0, max=25, step=1)
q_slider = labeled_slider('Influent flow, Q (MGD)', value=20, min=1, max=50, step=1)
s0_slider = labeled_slider('Influent substrate COD, S0 (mg/L)', value=300, min=100, max=600, step=10)

out = widgets.Output()

def update(change=None):
    SRT, Q, S_0 = srt_slider.value, q_slider.value, s0_slider.value
    result = simulate_asp(Q=Q, S_0=S_0, SRT=SRT)

    # The unit rounds SRT to the nearest whole day internally, so it's a
    # genuine step function, not a smooth curve -- sweep whole days only
    # rather than implying finer resolution than the model actually has.
    # SRT=0 is included deliberately: with this module's default kinetics,
    # SRT_min = 2.5 d, so SRT=0,1,2 all wash out -- a clearly visible
    # region, not just a single edge value.
    sweep_max = max(25, SRT + 5)
    sweep_srt = range(0, int(sweep_max) + 1)
    srt_vals, cod_vals = effluent_cod_curve(sweep_srt, Q=Q, S_0=S_0)

    with out:
        out.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(6, 4.5))
        plot_srt_curve(ax, srt_vals, cod_vals, result['SRT_min'],
                        result['SRT'], result['effluent_COD'], result['washed_out'])
        plt.show()
        print(format_readout(result))
        impact_text = format_impact_comparisons(result)
        if impact_text:
            print()
            print(impact_text)

for slider in (srt_slider, q_slider, s0_slider):
    slider.observe(update, names='value')
update()

ui = widgets.VBox([srt_slider, q_slider, s0_slider])
widgets.HBox([ui, out])

## Try it yourself

- At what SRT does the system wash out for the default influent? Does that match the vertical dashed line (`SRT_min`) on the plot?
- How does increasing the influent substrate concentration `S0` change the curve's shape? Does the washout SRT change?
- Real full-scale activated sludge plants typically operate at 5–20× (or more) above `SRT_min`, even though effluent COD looks "good enough" much closer to `SRT_min`. Why operate so far above the minimum? (Hint: this module only models BOD/COD removal — what other treatment goals or practical constraints might require a longer SRT?)
- HRT and reactor volume both come out of the SRT you pick (via the unit's internal sizing). Does the readout's HRT increase or decrease as you raise SRT, holding `Q` fixed? Does that match your intuition for why HRT and SRT are related but not the same thing?
- Try holding `SRT` fixed and only moving the `S0` (influent strength) slider. Effluent COD barely moves compared to how much it moves when you drag the `SRT` slider — SRT dominates. The theoretical reason (Rittmann & McCarty): at steady state, effluent *substrate* concentration in a CSTR-with-recycle is set by SRT and kinetics alone (your chosen SRT pins the biomass growth rate via `μ = 1/SRT + b`, and Monod kinetics then pins the one substrate concentration that produces that growth rate) — loading doesn't come into that equation. What mostly changes as you raise `S0` instead? Watch `HRT` and `WAS production` in the readout — a stronger influent shows up mainly as more sludge produced and a larger reactor needed, not as worse water leaving the plant.
